## 1. Normal Torch -> StableHLO using JAX

In [2]:
# Since torch version is 2.2, I ran the script with my local machine.

"""
from torch.export import export
import torch
from torch import nn
import torchax as tx
import torchax.export

input = torch.tensor([1.0, 2.0], dtype=torch.float32)
weight = torch.tensor([3.0, 4.0], dtype=torch.float32)
class MatMul(nn.Module):
    def forward(self, x, w):
        return torch.add(x, w)
model = MatMul()
exported = export(
    model,
    args=(input, weight),
)

weights, stablehlo = tx.export.exported_program_to_stablehlo(exported)
print(stablehlo.mlir_module())
"""
from pathlib import Path
path = Path("/workspace/PyTorchSim/TensorFlow/docs/PyTorchSimDocs/torch_stablehlo_add.mlir")
with path.open("r", encoding="utf-8") as f:
    print(f.read())

module @jit_func attributes {jax.uses_shape_polymorphism = false, mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<2xf32>, %arg1: tensor<2xf32>) -> (tensor<2xf32> {jax.result_info = "result[0]"}) {
    %cst = stablehlo.constant dense<1.000000e+00> : tensor<f32>
    %0 = stablehlo.broadcast_in_dim %cst, dims = [] : (tensor<f32>) -> tensor<2xf32>
    %1 = stablehlo.multiply %arg1, %0 : tensor<2xf32>
    %2 = stablehlo.add %arg0, %1 : tensor<2xf32>
    return %2 : tensor<2xf32>
  }
}


## 2. Tensorflow -> StableHLO Using XLA Compiler

In [1]:
import os
import tensorflow as tf
from pathlib import Path

STABLEHLO_OPT = "/workspace/stablehlo/build/bin/stablehlo-opt"
OUT_DIR = "out"

@tf.function(jit_compile=True)
def add_fn(x, y):
    return x + y

x = tf.constant([1.0, 2.0], tf.float32)
y = tf.constant([3.0, 4.0], tf.float32)

ir = add_fn.experimental_get_compiler_ir(x, y)(stage="stablehlo")

input_mlir = Path(OUT_DIR) / "input.mlir"
input_mlir.write_text(ir)
torout = Path(OUT_DIR) / "torch.mlir"

os.system(
    f"{STABLEHLO_OPT} "
    f"/workspace/PyTorchSim/TensorFlow/docs/PyTorchSimDocs/torch_stablehlo_add.mlir "
    f"--canonicalize --cse --stablehlo-target-independent-optimization -o {torout}"
)

print(torout.read_text())


2026-02-04 02:34:16.832166: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-04 02:34:40.675050: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
2026-02-04 02:34:40.912218: I external/local_xla/xla/service/service.cc:163] XLA service 0x1876d640 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
2026-02-04 02:34:40.912258: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): Host, Default Version


module @jit_func attributes {jax.uses_shape_polymorphism = false, mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<2xf32>, %arg1: tensor<2xf32>) -> (tensor<2xf32> {jax.result_info = "result[0]"}) {
    %0 = stablehlo.add %arg0, %arg1 : tensor<2xf32>
    return %0 : tensor<2xf32>
  }
}




## 2.1 After mlir-opt 

## 3. torch -> MLIR Using PyTorchSim's Custom TorchInductor Backend

In [ ]:
import torch, os, sys, subprocess, re, io
from pathlib import Path
from contextlib import redirect_stdout

f = io.StringIO()
with redirect_stdout(f):
    base_dir = os.environ.get("TORCHSIM_DIR", "/workspace/PyTorchSim")
    sys.path.append(base_dir)
    from Scheduler.scheduler import PyTorchSimRunner
    device = PyTorchSimRunner.setup_device().custom_device()

    a = torch.tensor([1.0], dtype=torch.float32, device=device)
    b = torch.tensor([3.0, 4.0], dtype=torch.float32, device=device)

    def add(x, y):
        return torch.add(x, y)

    opt_fn = torch.compile(dynamic=False)(add)
    _ = opt_fn(a, b)

stdout = f.getvalue()
m = re.search(r"Wrapper Codegen Path = (.+)", stdout)
wrapper_path = Path(m.group(1).strip())
code = wrapper_path.read_text()
mlir_match = re.search(r"custom_async_compile\.mlir\(\s*'''(.*?)'''\s*,?",code,re.DOTALL,)
print(mlir_match.group(1))


Using /root/.cache/torch_extensions/py310_cu121 as PyTorch extensions root...
No modifications detected for re-loaded extension module npu, skipping build step...
Loading extension module npu...


memref.global @buf0_spad : memref<2xf32, 1>
memref.global @buf1_spad : memref<2xf32, 1>
memref.global @buf2_spad : memref<2xf32, 1>
func.func @kernel(%in_ptr0: memref<2xf32>,
                       %in_ptr1: memref<2xf32>,
                       %out_ptr0: memref<2xf32>)
{
    %const0 = arith.constant 0 : index
    %const1 = arith.constant 2 : index
    %const2 = arith.constant 3 : index
    %alloc0 = memref.alloc() : memref<1xi32> // 0
    %alloc1 = memref.alloc() : memref<1xi32> // 1
    %alloc2 = memref.alloc() : memref<1xi32> // 2
    %spad0 = memref.get_global @buf0_spad : memref<2xf32, 1>
    %spad1 = memref.get_global @buf1_spad : memref<2xf32, 1>
    %spad2 = memref.get_global @buf2_spad : memref<2xf32, 1>
    affine.for %index0 = 0 to 2 step 2
    {
        memref.dma_start %in_ptr0[%index0], %spad0[%const0], %const1, %alloc0[%const0], %const0, %const1 : memref<2xf32>, memref<2xf32, 1>, memref<1xi32> {dram_stride=[1], sram_stride=[1], padding=0}
        memref.dma_start %in_pt

In [ ]:
import torch, os, sys, subprocess, re, io
from pathlib import Path
from contextlib import redirect_stdout

f = io.StringIO()
with redirect_stdout(f):
    base_dir = os.environ.get("TORCHSIM_DIR", "/workspace/PyTorchSim")
    sys.path.append(base_dir)
    from Scheduler.scheduler import PyTorchSimRunner
    device = PyTorchSimRunner.setup_device().custom_device()

    a = torch.tensor([1.0, 2.0], dtype=torch.float32, device=device)
    b = torch.tensor([3.0, 4.0], dtype=torch.float32, device=device)

    def add(a,b):
        return torch.add(a,b)

    opt_fn = torch.compile(dynamic=False)(add)
    _ = opt_fn(a,b)

stdout = f.getvalue()
m = re.search(r"Wrapper Codegen Path = (.+)", stdout)
wrapper_path = Path(m.group(1).strip())
code = wrapper_path.read_text()
mlir_match = re.search(r"custom_async_compile\.mlir\(\s*'''(.*?)'''\s*,?",code,re.DOTALL,)
print(mlir_match.group(1))


Using /root/.cache/torch_extensions/py310_cu121 as PyTorch extensions root...
No modifications detected for re-loaded extension module npu, skipping build step...
Loading extension module npu...


memref.global @buf0_spad : memref<1x512xf32, 1>
memref.global @buf1_spad : memref<1x512xf32, 1>
memref.global @buf2_spad : memref<1xf32, 1>
func.func @kernel(%in_ptr0: memref<2xf32>,
                       %in_ptr1: memref<2xf32>,
                       %out_ptr0: memref<1xf32>)
{
    %const0 = arith.constant 0 : index
    %const1 = arith.constant 1 : index
    %const2 = arith.constant 512 : index
    %const3 = arith.constant 2 : index
    %const4 = arith.constant 0.0 : f32
    %const5 = vector.broadcast %const4 : f32 to vector<8xf32>
    %const6 = arith.constant 3 : index
    %alloc0 = memref.alloc() : memref<1xi32> // 0
    %alloc1 = memref.alloc() : memref<1xi32> // 1
    %alloc2 = memref.alloc() : memref<1xi32> // 2
    %spad0 = memref.get_global @buf0_spad : memref<1x512xf32, 1>
    %spad1 = memref.get_global @buf1_spad : memref<1x512xf32, 1>
    %spad2 = memref.get_global @buf2_spad : memref<1xf32, 1>
    affine.for %dummy = 0 to 1 step 1
    {
        %tmp_acc0 = affine.for %ind